# RMT: 공개 Copy 조건 검증 (1단계)
T4 GPU에서 RMT만 실행합니다. V12 · source24 · answer48 · N=M24 · 4층/D128/FF256,
batch512 · LR1e-4 · FP32. 최대25K update이며 validation의 CE와 자기회귀 생성 기준을
함께 만족하면 종료합니다. GRT 적용은 이 결과 확인 후 진행합니다.

현재 수정본이 반영된 checkout에서 실행하세요. 설정은
`configs/rmt_paper_copy.yaml`, 실행과 결과 안내는 `README.md`를 참고하세요.

In [ ]:
from pathlib import Path
import sys, subprocess
import torch

REPO_DIR = Path('/content/GRT')
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
RUN_DIR = (Path('/content/drive/MyDrive/GRT/runs/rmt_paper_copy_fp32_b512_seed1234')
           if USE_DRIVE else REPO_DIR / 'runs/rmt_paper_copy_fp32_b512_seed1234')
assert (REPO_DIR / 'configs/rmt_paper_copy.yaml').exists(), '현재 수정본이 반영된 checkout이 필요합니다.'
assert torch.cuda.is_available(), 'GPU runtime이 필요합니다.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
# 현재 커널에도 src를 등록: editable install의 .pth는 커널 재시작 전에는 적용되지 않을 수 있습니다.
sys.path.insert(0, str(REPO_DIR / 'src'))
from grt.config import load_config  # 다음 셀 실행 전에 import 확인
print(torch.cuda.get_device_name(0), 'run:', RUN_DIR)

In [ ]:
# 새 학습 또는 동일 설정의 최신 checkpoint에서 재개.
from dataclasses import asdict
from grt.config import load_config
from grt.checkpoint import load_checkpoint

expected = asdict(load_config(REPO_DIR / 'configs/rmt_paper_copy.yaml', output_dir=RUN_DIR))
expected.pop('run')
candidates = []
for name in ('last.pt', 'best.pt', 'converged.pt'):
    path = RUN_DIR / name
    if path.exists():
        state = load_checkpoint(path)
        saved = dict(state['config'])
        saved.pop('run')
        if saved != expected:
            raise ValueError('설정이 다른 checkpoint입니다. 새 RUN_DIR을 사용하세요.')
        candidates.append((state['global_step'], bool(state.get('converged')), path))

command = [sys.executable, '-u', 'scripts/train.py', '--device', 'cuda']
if candidates:
    step, converged, checkpoint = max(candidates, key=lambda x: (x[0], x[1]))
    if converged or step >= expected['training']['max_steps']:
        print('학습 완료:', step, 'converged:', converged)
        if not (RUN_DIR / 'evaluation.json').exists():
            selected = (RUN_DIR / 'converged.pt' if converged and (RUN_DIR / 'converged.pt').exists()
                        else checkpoint if converged else RUN_DIR / 'best.pt')
            subprocess.run([sys.executable, '-u', 'scripts/evaluate.py', '--checkpoint', str(selected),
                            '--device', 'cuda'], cwd=REPO_DIR, check=True)
        command = None
    else:
        command += ['--resume', str(checkpoint)]
else:
    command += ['--config', 'configs/rmt_paper_copy.yaml', '--output-dir', str(RUN_DIR)]
if command:
    subprocess.run(command, cwd=REPO_DIR, check=True)

In [ ]:
# 학습 곡선과 공유할 결과.
import json
import matplotlib.pyplot as plt

validation, training = [], []
with (RUN_DIR / 'metrics.jsonl').open() as handle:
    for line in handle:
        row = json.loads(line)
        if 'val/loss' in row:
            validation.append(row)
        if 'train/loss' in row and (row['global_step'] == 1 or row['global_step'] % 100 == 0):
            training.append(row)
# 중단 후 재개 시 같은 step의 기록은 최신 값으로 표시.
validation = sorted({r['global_step']: r for r in validation}.values(), key=lambda r: r['global_step'])
training = sorted({r['global_step']: r for r in training}.values(), key=lambda r: r['global_step'])
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot([r['global_step'] for r in training], [r['train/loss'] for r in training], label='train CE', alpha=.6)
axes[0].plot([r['global_step'] for r in validation], [r['val/loss'] for r in validation], label='validation CE')
for key, label in [('val/token_accuracy', 'teacher forcing'),
                   ('val/autoregressive/token_accuracy', 'autoregressive'),
                   ('val/autoregressive/exact_match', 'autoregressive exact match')]:
    axes[1].plot([r['global_step'] for r in validation], [r[key] for r in validation], label=label)
axes[1].set_ylim(0, 1.02)
for ax in axes:
    ax.set_xlabel('optimizer updates')
    ax.legend()
    ax.grid(alpha=.2)
fig.tight_layout()
fig.savefig(RUN_DIR / 'convergence.png', dpi=150)
plt.show()

evaluation = json.loads((RUN_DIR / 'evaluation.json').read_text())
states = [load_checkpoint(RUN_DIR / name) for name in ('last.pt', 'best.pt', 'converged.pt')
          if (RUN_DIR / name).exists()]
state = max(states, key=lambda x: (x['global_step'], bool(x.get('converged'))))
summary = {'completed_updates': state['global_step'], 'validation_converged': state.get('converged', False),
           'best_validation_ce': state['best_validation_loss'],
           'last_validation': validation[-1] if validation else None, 'test': evaluation,
           'note': 'loss/token_accuracy/exact_match는 teacher forcing; autoregressive/*는 실제 생성 지표.'}
(RUN_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False))
print(json.dumps(summary, indent=2, ensure_ascii=False))
print('공유: summary.json, convergence.png, metrics.jsonl, evaluation.json, resolved_config.yaml, metadata.json')